# Advanced `defaultdict` Problems with Solutions

This notebook is an expanded practice set built around Python's `collections.defaultdict`.

## What this notebook emphasizes

- choosing the correct default factory;
- understanding when missing-key access mutates a `defaultdict`;
- counting, grouping, indexing, graph building, and sparse data;
- nested `defaultdict` structures;
- safe conversion back to plain dictionaries;
- stateful and non-deterministic factories;
- factory pitfalls involving shared mutable objects;
- decorators and function-call statistics;
- testing invariants and edge cases;
- reasoning about time and space complexity.

All examples use only the Python standard library.

> **Best practice:** use `defaultdict` when "missing key means create a fresh default value" is genuinely part of the data model. If a missing key is exceptional, a normal `dict` may communicate intent better.

In [1]:
from collections import defaultdict, namedtuple
from functools import partial, wraps
from datetime import datetime, timezone
from pprint import pprint
from typing import Iterable, Hashable

## Quick refresher

A `defaultdict(factory)` behaves like a normal dictionary except that indexing a missing key with `d[key]` calls `factory()`, stores the returned value under `key`, and returns it.

Important consequences:

1. `d[key]` can **mutate** the dictionary when `key` is absent.
2. `d.get(key)` does **not** invoke the default factory.
3. The factory must be callable with **no arguments**.
4. Mutable factories such as `list`, `set`, and `dict` are useful because each missing key receives a fresh object.

In [2]:
# Basic behavior: indexing mutates on a miss.
d = defaultdict(int)

assert "missing" not in d
value = d["missing"]

assert value == 0
assert d == {"missing": 0}

# .get() does not invoke the factory.
before = dict(d)
assert d.get("another") is None
assert dict(d) == before

print("Basic defaultdict semantics verified.")

Basic defaultdict semantics verified.


# Problem 1 — Robust frequency table

Write `frequency_table(items)` that counts arbitrary hashable values.

### Requirements
- Use `defaultdict`.
- Return a regular `dict`.
- Preserve the natural insertion order of first appearance.
- Handle an empty iterable.

### Example
`["a", "b", "a", "c", "b", "a"] -> {"a": 3, "b": 2, "c": 1}`

In [3]:
def frequency_table(items: Iterable[Hashable]) -> dict:
    counts = defaultdict(int)
    for item in items:
        counts[item] += 1
    return dict(counts)


assert frequency_table([]) == {}
assert frequency_table(["a", "b", "a", "c", "b", "a"]) == {
    "a": 3,
    "b": 2,
    "c": 1,
}
assert frequency_table([1, 1, 2, None, None]) == {
    1: 2,
    2: 1,
    None: 2,
}

print(frequency_table("mississippi"))

{'m': 1, 'i': 4, 's': 4, 'p': 2}


### Solution notes

`int()` returns `0`, so `defaultdict(int)` directly models the invariant "an unseen item has count zero."

**Complexity:** expected `O(n)` time and `O(k)` space, where `k` is the number of distinct keys.

# Problem 2 — Group records by a computed key

Given employee records, group employee names by department.

### Requirements
- Missing department values must go under `"UNKNOWN"`.
- Do not create unnecessary intermediate lists manually.
- Sort each department's names before returning.
- Return a normal dictionary.

In [4]:
employees = [
    {"name": "Ava", "department": "Engineering"},
    {"name": "Ben", "department": "Finance"},
    {"name": "Cara", "department": "Engineering"},
    {"name": "Dion"},
    {"name": "Eli", "department": "Finance"},
    {"name": "Faye"},
]


def group_names_by_department(records):
    grouped = defaultdict(list)

    for record in records:
        department = record.get("department", "UNKNOWN")
        grouped[department].append(record["name"])

    return {
        department: sorted(names)
        for department, names in grouped.items()
    }


result = group_names_by_department(employees)
pprint(result)

assert result == {
    "Engineering": ["Ava", "Cara"],
    "Finance": ["Ben", "Eli"],
    "UNKNOWN": ["Dion", "Faye"],
}

{'Engineering': ['Ava', 'Cara'],
 'Finance': ['Ben', 'Eli'],
 'UNKNOWN': ['Dion', 'Faye']}


# Problem 3 — Inverted index

Build an inverted index that maps each normalized word to the set of document IDs containing that word.

### Requirements
- Normalize to lowercase.
- Treat punctuation `.,!?;:` as separators.
- Store document IDs in a `set` to avoid duplicates.
- Return `dict[str, set[int]]`.

In [5]:
documents = {
    101: "Python is expressive, and Python is practical.",
    102: "Practical data structures matter!",
    103: "Python data workflows are expressive.",
}


def tokenize(text: str) -> list[str]:
    cleaned = text.lower()
    for ch in ".,!?;:":
        cleaned = cleaned.replace(ch, " ")
    return cleaned.split()


def build_inverted_index(docs):
    index = defaultdict(set)

    for doc_id, text in docs.items():
        for word in tokenize(text):
            index[word].add(doc_id)

    return dict(index)


index = build_inverted_index(documents)
pprint(index)

assert index["python"] == {101, 103}
assert index["practical"] == {101, 102}
assert index["data"] == {102, 103}

{'and': {101},
 'are': {103},
 'data': {102, 103},
 'expressive': {101, 103},
 'is': {101},
 'matter': {102},
 'practical': {101, 102},
 'python': {101, 103},
 'structures': {102},
 'workflows': {103}}


### Why `defaultdict(set)`?

Using `list` would require duplicate handling. A `set` expresses the real invariant: a document either contains a word or it does not.

# Problem 4 — Nested aggregation

Aggregate transactions by `region` and then by `product`.

Return total revenue for every `(region, product)` pair.

### Input fields
- `region`
- `product`
- `quantity`
- `unit_price`

Use a nested `defaultdict`.

In [6]:
transactions = [
    {"region": "EU", "product": "A", "quantity": 2, "unit_price": 10.0},
    {"region": "EU", "product": "B", "quantity": 1, "unit_price": 25.0},
    {"region": "EU", "product": "A", "quantity": 3, "unit_price": 10.0},
    {"region": "US", "product": "A", "quantity": 5, "unit_price": 12.0},
    {"region": "US", "product": "C", "quantity": 2, "unit_price": 40.0},
]


def revenue_by_region_and_product(rows):
    totals = defaultdict(lambda: defaultdict(float))

    for row in rows:
        revenue = row["quantity"] * row["unit_price"]
        totals[row["region"]][row["product"]] += revenue

    return {
        region: dict(product_totals)
        for region, product_totals in totals.items()
    }


revenue = revenue_by_region_and_product(transactions)
pprint(revenue)

assert revenue["EU"]["A"] == 50.0
assert revenue["EU"]["B"] == 25.0
assert revenue["US"]["A"] == 60.0
assert revenue["US"]["C"] == 80.0

{'EU': {'A': 50.0, 'B': 25.0}, 'US': {'A': 60.0, 'C': 80.0}}


# Problem 5 — Adjacency list for an undirected graph

Given undirected edges, construct a graph where each node maps to a set of neighbors.

Then write:
- `degree(graph, node)`
- `isolated_nodes(graph)`

A supplied `all_nodes` collection may contain nodes that never appear in an edge.

In [7]:
edges = [
    ("A", "B"),
    ("A", "C"),
    ("B", "C"),
    ("D", "E"),
]
all_nodes = {"A", "B", "C", "D", "E", "F"}


def build_undirected_graph(edges, all_nodes=()):
    graph = defaultdict(set)

    for node in all_nodes:
        graph[node]

    for left, right in edges:
        graph[left].add(right)
        graph[right].add(left)

    return graph


def degree(graph, node):
    return len(graph.get(node, ()))


def isolated_nodes(graph):
    return {node for node, neighbors in graph.items() if not neighbors}


graph = build_undirected_graph(edges, all_nodes)

pprint(dict(graph))
assert degree(graph, "A") == 2
assert degree(graph, "F") == 0
assert degree(graph, "UNKNOWN") == 0
assert "UNKNOWN" not in graph
assert isolated_nodes(graph) == {"F"}

{'A': {'C', 'B'},
 'B': {'A', 'C'},
 'C': {'A', 'B'},
 'D': {'E'},
 'E': {'D'},
 'F': set()}


### Key subtlety

`graph["UNKNOWN"]` would create a new key because indexing invokes the factory. `graph.get("UNKNOWN", ())` avoids that mutation.

This distinction matters in read-only queries.

# Problem 6 — Sparse matrix accumulation

Represent a sparse matrix as a mapping `(row, column) -> value`.

Process updates of the form `(row, column, delta)` and return only non-zero entries.

Use `defaultdict(float)`.

In [8]:
updates = [
    (0, 0, 3.5),
    (0, 2, 1.0),
    (0, 0, -1.5),
    (2, 1, 4.0),
    (0, 2, -1.0),
]


def accumulate_sparse_matrix(updates):
    matrix = defaultdict(float)

    for row, col, delta in updates:
        matrix[(row, col)] += delta

    return {
        coordinate: value
        for coordinate, value in matrix.items()
        if value != 0
    }


sparse = accumulate_sparse_matrix(updates)
print(sparse)

assert sparse == {
    (0, 0): 2.0,
    (2, 1): 4.0,
}

{(0, 0): 2.0, (2, 1): 4.0}


# Problem 7 — Session event statistics

For each user, compute:
- number of events;
- total duration;
- set of event types.

Use a nested structure created by a factory function.

In [9]:
events = [
    {"user": "u1", "event": "view", "duration": 1.2},
    {"user": "u1", "event": "click", "duration": 0.4},
    {"user": "u2", "event": "view", "duration": 2.0},
    {"user": "u1", "event": "view", "duration": 0.8},
]


def new_user_stats():
    return {
        "count": 0,
        "total_duration": 0.0,
        "event_types": set(),
    }


def aggregate_user_events(events):
    stats = defaultdict(new_user_stats)

    for event in events:
        record = stats[event["user"]]
        record["count"] += 1
        record["total_duration"] += event["duration"]
        record["event_types"].add(event["event"])

    return dict(stats)


stats = aggregate_user_events(events)
pprint(stats)

assert stats["u1"]["count"] == 3
assert abs(stats["u1"]["total_duration"] - 2.4) < 1e-12
assert stats["u1"]["event_types"] == {"view", "click"}

{'u1': {'count': 3,
        'event_types': {'click', 'view'},
        'total_duration': 2.4000000000000004},
 'u2': {'count': 1, 'event_types': {'view'}, 'total_duration': 2.0}}


# Problem 8 — Factory side effects and read semantics

A factory increments a counter every time it is called.

Determine which operations invoke the factory:
- indexing a missing key;
- `.get()` on a missing key;
- membership testing with `in`;
- `.setdefault()`;
- indexing an existing key.

Then verify the behavior programmatically.

In [10]:
factory_calls = 0


def instrumented_factory():
    global factory_calls
    factory_calls += 1
    return []


d = defaultdict(instrumented_factory)

d["a"].append(1)
assert factory_calls == 1

assert d.get("b") is None
assert factory_calls == 1
assert "b" not in d

assert ("c" in d) is False
assert factory_calls == 1

returned = d.setdefault("d", ["manual"])
assert returned == ["manual"]
assert factory_calls == 1

assert d["a"] == [1]
assert factory_calls == 1

print("factory_calls =", factory_calls)
print(dict(d))

factory_calls = 1
{'a': [1], 'd': ['manual']}


### Takeaway

`defaultdict`'s factory is tied specifically to missing-key handling through `__getitem__` (`d[key]`), not every dictionary lookup-like method.

# Problem 9 — The shared-mutable-default trap

A common mistake is accidentally returning the **same mutable object** for every missing key.

1. Demonstrate the bug.
2. Fix it.
3. Explain why `defaultdict(list)` is safer than a factory that closes over one shared list.

In [11]:
shared = []
bad = defaultdict(lambda: shared)

bad["x"].append("from x")
bad["y"].append("from y")

print("bad['x']:", bad["x"])
print("bad['y']:", bad["y"])

assert bad["x"] is bad["y"]
assert bad["x"] == ["from x", "from y"]


good = defaultdict(list)
good["x"].append("from x")
good["y"].append("from y")

print("good['x']:", good["x"])
print("good['y']:", good["y"])

assert good["x"] is not good["y"]
assert good["x"] == ["from x"]
assert good["y"] == ["from y"]

bad['x']: ['from x', 'from y']
bad['y']: ['from x', 'from y']
good['x']: ['from x']
good['y']: ['from y']


### Best practice

Factories for mutable values should usually **construct a fresh object per missing key**.

Good:
```python
defaultdict(list)
defaultdict(dict)
defaultdict(set)
defaultdict(lambda: {"count": 0})
```

Risky:
```python
shared = []
defaultdict(lambda: shared)
```

# Problem 10 — Constant defaults using a factory

Create a dictionary whose missing keys receive the string `"UNKNOWN"`.

Then initialize it with existing keyword arguments.

Finally, show how `functools.partial` can create a reusable constructor.

In [12]:
unknown_dict = partial(defaultdict, lambda: "UNKNOWN")

person = unknown_dict(
    name="Ava",
    age=31,
)

assert person["name"] == "Ava"
assert person["country"] == "UNKNOWN"
assert "country" in person

print(dict(person))

{'name': 'Ava', 'age': 31, 'country': 'UNKNOWN'}


# Problem 11 — Hierarchical log aggregation

Each log record has:
- `service`
- `level`
- `message`

Build a structure:

`service -> level -> list of messages`

Use nested `defaultdict` factories.

In [13]:
logs = [
    {"service": "auth", "level": "INFO", "message": "login ok"},
    {"service": "auth", "level": "ERROR", "message": "bad token"},
    {"service": "billing", "level": "INFO", "message": "invoice created"},
    {"service": "auth", "level": "INFO", "message": "logout"},
]


def aggregate_logs(logs):
    grouped = defaultdict(lambda: defaultdict(list))

    for record in logs:
        grouped[record["service"]][record["level"]].append(record["message"])

    return {
        service: {
            level: list(messages)
            for level, messages in levels.items()
        }
        for service, levels in grouped.items()
    }


grouped_logs = aggregate_logs(logs)
pprint(grouped_logs)

assert grouped_logs["auth"]["INFO"] == ["login ok", "logout"]
assert grouped_logs["auth"]["ERROR"] == ["bad token"]
assert grouped_logs["billing"]["INFO"] == ["invoice created"]

{'auth': {'ERROR': ['bad token'], 'INFO': ['login ok', 'logout']},
 'billing': {'INFO': ['invoice created']}}


# Problem 12 — Recursive tree

Create an arbitrarily deep tree using a recursive default factory.

Then store values at paths such as:
- `config["database"]["primary"]["host"]`
- `config["database"]["primary"]["port"]`
- `config["features"]["search"]["enabled"]`

Finally, write a recursive function that converts the tree to plain dictionaries.

In [14]:
def tree():
    return defaultdict(tree)


config = tree()
config["database"]["primary"]["host"] = "db.internal"
config["database"]["primary"]["port"] = 5432
config["features"]["search"]["enabled"] = True


def to_plain_dict(value):
    if isinstance(value, defaultdict):
        return {
            key: to_plain_dict(child)
            for key, child in value.items()
        }
    return value


plain_config = to_plain_dict(config)
pprint(plain_config)

assert plain_config == {
    "database": {
        "primary": {
            "host": "db.internal",
            "port": 5432,
        }
    },
    "features": {
        "search": {
            "enabled": True,
        }
    },
}

{'database': {'primary': {'host': 'db.internal', 'port': 5432}},
 'features': {'search': {'enabled': True}}}


### Caution

Recursive `defaultdict` trees are convenient for building data, but accidental reads such as `config["typo"]["x"]` silently create branches. They are best used in controlled construction code.

# Problem 13 — Word-position index

Map each normalized word to **all positions** where it appears in a sequence.

Example:

`["to", "be", "or", "not", "to", "be"]`

should produce:

`{"to": [0, 4], "be": [1, 5], "or": [2], "not": [3]}`

In [15]:
def position_index(words):
    positions = defaultdict(list)

    for index, word in enumerate(words):
        positions[word.lower()].append(index)

    return dict(positions)


words = ["To", "be", "or", "not", "to", "BE"]
positions = position_index(words)
print(positions)

assert positions == {
    "to": [0, 4],
    "be": [1, 5],
    "or": [2],
    "not": [3],
}

{'to': [0, 4], 'be': [1, 5], 'or': [2], 'not': [3]}


# Problem 14 — Multi-metric numeric aggregation

For each category, calculate:
- count;
- sum;
- minimum;
- maximum;
- mean.

Use a `defaultdict` whose factory returns a fresh state record.

Avoid storing every observation.

In [16]:
measurements = [
    ("cpu", 0.70),
    ("memory", 0.55),
    ("cpu", 0.80),
    ("cpu", 0.60),
    ("memory", 0.65),
]


def metric_state():
    return {
        "count": 0,
        "sum": 0.0,
        "min": float("inf"),
        "max": float("-inf"),
    }


def aggregate_metrics(rows):
    data = defaultdict(metric_state)

    for category, value in rows:
        state = data[category]
        state["count"] += 1
        state["sum"] += value
        state["min"] = min(state["min"], value)
        state["max"] = max(state["max"], value)

    result = {}
    for category, state in data.items():
        result[category] = {
            **state,
            "mean": state["sum"] / state["count"],
        }

    return result


metric_summary = aggregate_metrics(measurements)
pprint(metric_summary)

assert metric_summary["cpu"]["count"] == 3
assert abs(metric_summary["cpu"]["mean"] - 0.7) < 1e-12
assert metric_summary["memory"]["min"] == 0.55
assert metric_summary["memory"]["max"] == 0.65

{'cpu': {'count': 3,
         'max': 0.8,
         'mean': 0.7000000000000001,
         'min': 0.6,
         'sum': 2.1},
 'memory': {'count': 2,
            'max': 0.65,
            'mean': 0.6000000000000001,
            'min': 0.55,
            'sum': 1.2000000000000002}}


# Problem 15 — Directed graph indegree and outdegree

Given directed edges `(source, target)`, compute both:
- `outdegree[node]`
- `indegree[node]`

Every node appearing anywhere must be present in both returned dictionaries, even if its degree is zero.

In [17]:
directed_edges = [
    ("A", "B"),
    ("A", "C"),
    ("C", "B"),
    ("D", "C"),
]


def degrees(edges):
    indegree = defaultdict(int)
    outdegree = defaultdict(int)

    for source, target in edges:
        indegree[source]
        outdegree[target]

        outdegree[source] += 1
        indegree[target] += 1

    return dict(indegree), dict(outdegree)


indegree, outdegree = degrees(directed_edges)

print("indegree:", indegree)
print("outdegree:", outdegree)

assert indegree == {"A": 0, "B": 2, "C": 2, "D": 0}
assert outdegree == {"B": 0, "C": 1, "A": 2, "D": 1}

indegree: {'A': 0, 'B': 2, 'C': 2, 'D': 0}
outdegree: {'B': 0, 'A': 2, 'C': 1, 'D': 1}


# Problem 16 — Freeze a defaultdict

Sometimes construction benefits from default creation, but later reads should fail on unknown keys.

Build a frequency table using `defaultdict(int)`, then set:

```python
d.default_factory = None
```

Verify that existing keys still work and missing keys now raise `KeyError`.

In [18]:
frozen_counts = defaultdict(int)

for ch in "abracadabra":
    frozen_counts[ch] += 1

frozen_counts.default_factory = None

assert frozen_counts["a"] == 5

try:
    frozen_counts["z"]
except KeyError:
    print("Missing keys now raise KeyError as expected.")
else:
    raise AssertionError("Expected KeyError")

print(dict(frozen_counts))

Missing keys now raise KeyError as expected.
{'a': 5, 'b': 2, 'r': 2, 'c': 1, 'd': 1}


### Why this can be useful

A two-phase workflow is common:

1. **Build phase:** permissive automatic creation.
2. **Read phase:** strict lookup.

Disabling `default_factory` makes accidental late-stage mutations easier to catch.

# Problem 17 — Function-call statistics decorator

Create a utility that returns:
- a decorator;
- a shared statistics mapping.

For each decorated function, track:
- `count`;
- `first_called`;
- `last_called`.

The timestamp for a function must be created only when that function is first called.

In [19]:
StatsBundle = namedtuple("StatsBundle", "decorator data")


def function_stats():
    def initial_record():
        now = datetime.now(timezone.utc)
        return {
            "count": 0,
            "first_called": now,
            "last_called": now,
        }

    data = defaultdict(initial_record)

    def decorator(fn):
        @wraps(fn)
        def wrapper(*args, **kwargs):
            record = data[fn.__name__]
            record["count"] += 1
            record["last_called"] = datetime.now(timezone.utc)
            return fn(*args, **kwargs)

        return wrapper

    return StatsBundle(decorator, data)


stats = function_stats()


@stats.decorator
def add(x, y):
    return x + y


@stats.decorator
def multiply(x, y):
    return x * y


assert dict(stats.data) == {}
assert add(2, 3) == 5
assert add(10, 20) == 30
assert multiply(4, 5) == 20

pprint(dict(stats.data))

assert stats.data["add"]["count"] == 2
assert stats.data["multiply"]["count"] == 1
assert stats.data["add"]["first_called"] <= stats.data["add"]["last_called"]

{'add': {'count': 2,
         'first_called': datetime.datetime(2026, 8, 23, 11, 57, 15, 625009, tzinfo=datetime.timezone.utc),
         'last_called': datetime.datetime(2026, 8, 23, 11, 57, 15, 625053, tzinfo=datetime.timezone.utc)},
 'multiply': {'count': 1,
              'first_called': datetime.datetime(2026, 8, 23, 11, 57, 15, 625103, tzinfo=datetime.timezone.utc),
              'last_called': datetime.datetime(2026, 8, 23, 11, 57, 15, 625105, tzinfo=datetime.timezone.utc)}}


# Problem 18 — Detect accidental mutation caused by reads

Suppose a report function should be read-only.

The buggy version indexes missing keys and silently changes the dictionary.

1. Reproduce the bug.
2. Rewrite the function so it does not mutate.
3. Test the invariant `before == after`.

In [20]:
inventory = defaultdict(int, apples=5, oranges=2)


def buggy_lookup(stock, product):
    return stock[product]


before = dict(inventory)
assert buggy_lookup(inventory, "bananas") == 0
after = dict(inventory)

assert before != after
assert "bananas" in inventory

del inventory["bananas"]


def safe_lookup(stock, product):
    return stock.get(product, 0)


before = dict(inventory)
assert safe_lookup(inventory, "bananas") == 0
after = dict(inventory)

assert before == after
assert "bananas" not in inventory

print("Read-only invariant preserved.")

Read-only invariant preserved.


# Problem 19 — Build a co-occurrence matrix

Given baskets of items, count how often each unordered pair appears together.

Example basket:
`["bread", "milk", "eggs"]`

contributes to:
- bread ↔ milk
- bread ↔ eggs
- milk ↔ eggs

Return a nested dictionary where `matrix[a][b]` is the co-occurrence count.

In [21]:
baskets = [
    ["bread", "milk", "eggs"],
    ["bread", "milk"],
    ["milk", "eggs"],
    ["bread", "eggs"],
]


def cooccurrence_matrix(baskets):
    matrix = defaultdict(lambda: defaultdict(int))

    for basket in baskets:
        unique_items = sorted(set(basket))

        for i, left in enumerate(unique_items):
            for right in unique_items[i + 1:]:
                matrix[left][right] += 1
                matrix[right][left] += 1

    return {
        left: dict(neighbors)
        for left, neighbors in matrix.items()
    }


matrix = cooccurrence_matrix(baskets)
pprint(matrix)

assert matrix["bread"]["milk"] == 2
assert matrix["milk"]["bread"] == 2
assert matrix["bread"]["eggs"] == 2
assert matrix["milk"]["eggs"] == 2

{'bread': {'eggs': 2, 'milk': 2},
 'eggs': {'bread': 2, 'milk': 2},
 'milk': {'bread': 2, 'eggs': 2}}


# Problem 20 — Aggregate API-style responses with missing fields

You receive records where `country` or `status` may be absent.

Build:

`country -> status -> count`

Rules:
- missing country -> `"UNKNOWN_COUNTRY"`;
- missing status -> `"UNKNOWN_STATUS"`;
- return plain dictionaries.

In [22]:
responses = [
    {"country": "BG", "status": 200},
    {"country": "BG", "status": 500},
    {"country": "BG"},
    {"status": 200},
    {},
    {"country": "DE", "status": 200},
]


def response_summary(records):
    summary = defaultdict(lambda: defaultdict(int))

    for record in records:
        country = record.get("country", "UNKNOWN_COUNTRY")
        status = record.get("status", "UNKNOWN_STATUS")
        summary[country][status] += 1

    return {
        country: dict(status_counts)
        for country, status_counts in summary.items()
    }


summary = response_summary(responses)
pprint(summary)

assert summary["BG"] == {
    200: 1,
    500: 1,
    "UNKNOWN_STATUS": 1,
}
assert summary["UNKNOWN_COUNTRY"] == {
    200: 1,
    "UNKNOWN_STATUS": 1,
}
assert summary["DE"] == {200: 1}

{'BG': {200: 1, 500: 1, 'UNKNOWN_STATUS': 1},
 'DE': {200: 1},
 'UNKNOWN_COUNTRY': {200: 1, 'UNKNOWN_STATUS': 1}}


# Problem 21 — Compare dict.get, setdefault, and defaultdict

Implement the same grouping task three ways:

1. `dict.get`
2. `dict.setdefault`
3. `defaultdict(list)`

Use assertions to verify that all produce equivalent results.

In [23]:
pairs = [
    ("fruit", "apple"),
    ("fruit", "pear"),
    ("vegetable", "carrot"),
    ("fruit", "banana"),
]


def group_with_get(pairs):
    result = {}

    for key, value in pairs:
        values = result.get(key, [])
        values.append(value)
        result[key] = values

    return result


def group_with_setdefault(pairs):
    result = {}

    for key, value in pairs:
        result.setdefault(key, []).append(value)

    return result


def group_with_defaultdict(pairs):
    result = defaultdict(list)

    for key, value in pairs:
        result[key].append(value)

    return dict(result)


a = group_with_get(pairs)
b = group_with_setdefault(pairs)
c = group_with_defaultdict(pairs)

assert a == b == c
pprint(c)

{'fruit': ['apple', 'pear', 'banana'], 'vegetable': ['carrot']}


### Discussion

All three approaches can be valid.

- `get` is explicit but can be verbose when the updated value must be written back.
- `setdefault` is concise for isolated grouping operations.
- `defaultdict` is often clearest when the same missing-key policy applies throughout the whole mapping.

# Problem 22 — Two-level deduplication

You receive `(team, user_id, permission)` records, possibly with duplicates.

Build:

`team -> user_id -> set(permission)`

Then calculate the number of unique permissions per user.

In [24]:
permission_rows = [
    ("alpha", 1, "read"),
    ("alpha", 1, "write"),
    ("alpha", 1, "read"),
    ("alpha", 2, "read"),
    ("beta", 3, "admin"),
    ("beta", 3, "admin"),
]


def permissions_by_team(rows):
    permissions = defaultdict(lambda: defaultdict(set))

    for team, user_id, permission in rows:
        permissions[team][user_id].add(permission)

    return permissions


permissions = permissions_by_team(permission_rows)

permission_counts = {
    team: {
        user_id: len(values)
        for user_id, values in users.items()
    }
    for team, users in permissions.items()
}

pprint(permission_counts)

assert permission_counts == {
    "alpha": {1: 2, 2: 1},
    "beta": {3: 1},
}

{'alpha': {1: 2, 2: 1}, 'beta': {3: 1}}


# Problem 23 — Best-selling product per store

Aggregate sales by store and product, then return the product with the greatest quantity for each store.

If there is a tie, choose the lexicographically smaller product name.

In [25]:
sales = [
    ("S1", "apple", 3),
    ("S1", "banana", 4),
    ("S1", "apple", 2),
    ("S2", "pear", 5),
    ("S2", "apple", 5),
    ("S2", "pear", 1),
]


def best_seller_by_store(rows):
    totals = defaultdict(lambda: defaultdict(int))

    for store, product, quantity in rows:
        totals[store][product] += quantity

    result = {}

    for store, products in totals.items():
        result[store] = min(
            products,
            key=lambda product: (-products[product], product),
        )

    return result, totals


best, totals = best_seller_by_store(sales)

print("best:", best)
print("totals:")
pprint({store: dict(values) for store, values in totals.items()})

assert best == {
    "S1": "apple",
    "S2": "pear",
}

best: {'S1': 'apple', 'S2': 'pear'}
totals:
{'S1': {'apple': 5, 'banana': 4}, 'S2': {'apple': 5, 'pear': 6}}


# Problem 24 — Factory validation and defensive construction

A `defaultdict` factory must be callable without arguments.

Create a helper `make_defaultdict(factory)` that:
- rejects non-callables;
- checks whether the factory can be called with no arguments;
- returns a `defaultdict(factory)` if valid.

For this exercise, perform a trial call and wrap `TypeError` with a clearer message.

In [26]:
def make_defaultdict(factory):
    if not callable(factory):
        raise TypeError("factory must be callable")

    try:
        factory()
    except TypeError as exc:
        raise TypeError(
            "factory must be callable with no required arguments"
        ) from exc

    return defaultdict(factory)


assert isinstance(make_defaultdict(list), defaultdict)
assert isinstance(make_defaultdict(int), defaultdict)

try:
    make_defaultdict(123)
except TypeError as exc:
    print(exc)

try:
    make_defaultdict(lambda x: x)
except TypeError as exc:
    print(exc)

factory must be callable
factory must be callable with no required arguments


### Note

The trial-call technique is intentionally simple for this exercise. In production, invoking an arbitrary factory merely to validate it may have side effects. Often the best validation is simply to let `defaultdict` call the factory naturally when a missing key is first accessed.

# Problem 25 — Challenge: event stream with rolling categorical summaries

Given an event stream `(minute, category, value)`, build:

`minute -> category -> {"count": ..., "sum": ...}`

Then derive a second structure:

`category -> list[(minute, mean)]`

sorted by minute.

This problem combines nested factories, aggregation, conversion, and derived output.

In [27]:
stream = [
    (2, "latency", 120.0),
    (1, "latency", 100.0),
    (1, "errors", 1.0),
    (2, "latency", 140.0),
    (2, "errors", 0.0),
    (3, "errors", 2.0),
]


def new_bucket():
    return {"count": 0, "sum": 0.0}


def summarize_stream(stream):
    by_minute = defaultdict(lambda: defaultdict(new_bucket))

    for minute, category, value in stream:
        bucket = by_minute[minute][category]
        bucket["count"] += 1
        bucket["sum"] += value

    category_series = defaultdict(list)

    for minute in sorted(by_minute):
        for category, bucket in by_minute[minute].items():
            mean = bucket["sum"] / bucket["count"]
            category_series[category].append((minute, mean))

    plain_by_minute = {
        minute: {
            category: dict(bucket)
            for category, bucket in categories.items()
        }
        for minute, categories in by_minute.items()
    }

    return plain_by_minute, dict(category_series)


by_minute, category_series = summarize_stream(stream)

print("by_minute:")
pprint(by_minute)
print("\ncategory_series:")
pprint(category_series)

assert category_series["latency"] == [
    (1, 100.0),
    (2, 130.0),
]
assert category_series["errors"] == [
    (1, 1.0),
    (2, 0.0),
    (3, 2.0),
]

by_minute:
{1: {'errors': {'count': 1, 'sum': 1.0}, 'latency': {'count': 1, 'sum': 100.0}},
 2: {'errors': {'count': 1, 'sum': 0.0}, 'latency': {'count': 2, 'sum': 260.0}},
 3: {'errors': {'count': 1, 'sum': 2.0}}}

category_series:
{'errors': [(1, 1.0), (2, 0.0), (3, 2.0)], 'latency': [(1, 100.0), (2, 130.0)]}


# Problem 26 — Challenge: dependency graph and reverse graph

Given dependencies `(package, dependency)`, build both:

- forward graph: package -> dependencies
- reverse graph: dependency -> dependents

Use sets to eliminate duplicate edges.

Then implement `roots`, defined here as packages with no incoming edges.

In [28]:
dependencies = [
    ("app", "auth"),
    ("app", "db"),
    ("auth", "crypto"),
    ("auth", "db"),
    ("worker", "db"),
    ("app", "db"),
]


def build_dependency_graphs(edges):
    forward = defaultdict(set)
    reverse = defaultdict(set)
    nodes = set()

    for package, dependency in edges:
        nodes.update((package, dependency))
        forward[package].add(dependency)
        reverse[dependency].add(package)

    for node in nodes:
        forward[node]
        reverse[node]

    return forward, reverse


def roots(forward, reverse):
    return {
        node
        for node in forward
        if len(reverse[node]) == 0
    }


forward, reverse = build_dependency_graphs(dependencies)

print("forward:")
pprint(dict(forward))
print("\nreverse:")
pprint(dict(reverse))
print("\nroots:", roots(forward, reverse))

assert roots(forward, reverse) == {"app", "worker"}
assert forward["app"] == {"auth", "db"}
assert reverse["db"] == {"app", "auth", "worker"}

forward:
{'app': {'auth', 'db'},
 'auth': {'crypto', 'db'},
 'crypto': set(),
 'db': set(),
 'worker': {'db'}}

reverse:
{'app': set(),
 'auth': {'app'},
 'crypto': {'auth'},
 'db': {'worker', 'auth', 'app'},
 'worker': set()}

roots: {'worker', 'app'}


# Problem 27 — Challenge: nested factory created with partial

Create a reusable constructor for:

`defaultdict(lambda: "unknown")`

using `functools.partial`.

Use it to create several person records, then group names by eye color.

In [29]:
eye_record = partial(defaultdict, lambda: "unknown")

people = {
    "john": eye_record(age=20, eye_color="blue"),
    "jack": eye_record(age=25, eye_color="brown"),
    "jill": eye_record(age=22, eye_color="blue"),
    "eric": eye_record(age=35),
    "michael": eye_record(age=27),
}


def names_by_eye_color(people):
    grouped = defaultdict(list)

    for name, details in people.items():
        grouped[details["eye_color"]].append(name)

    return dict(grouped)


eye_groups = names_by_eye_color(people)
pprint(eye_groups)

assert eye_groups == {
    "blue": ["john", "jill"],
    "brown": ["jack"],
    "unknown": ["eric", "michael"],
}

{'blue': ['john', 'jill'], 'brown': ['jack'], 'unknown': ['eric', 'michael']}


# Problem 28 — Challenge: write a reusable nested_defaultdict

Write:

```python
nested_defaultdict(depth, leaf_factory)
```

such that:

- `depth=1` returns `defaultdict(leaf_factory)`;
- `depth=2` returns a `defaultdict` whose missing values are `defaultdict(leaf_factory)`;
- and so on.

Use it to build a three-level counter:
`country -> city -> event -> count`.

In [30]:
def nested_defaultdict(depth, leaf_factory):
    if depth < 1:
        raise ValueError("depth must be >= 1")

    if depth == 1:
        return defaultdict(leaf_factory)

    return defaultdict(
        lambda: nested_defaultdict(depth - 1, leaf_factory)
    )


events = [
    ("BG", "Sofia", "view"),
    ("BG", "Sofia", "view"),
    ("BG", "Sofia", "click"),
    ("BG", "Plovdiv", "view"),
    ("DE", "Berlin", "view"),
]

counter = nested_defaultdict(3, int)

for country, city, event in events:
    counter[country][city][event] += 1

assert counter["BG"]["Sofia"]["view"] == 2
assert counter["BG"]["Sofia"]["click"] == 1
assert counter["BG"]["Plovdiv"]["view"] == 1
assert counter["DE"]["Berlin"]["view"] == 1

pprint(to_plain_dict(counter))

{'BG': {'Plovdiv': {'view': 1}, 'Sofia': {'click': 1, 'view': 2}},
 'DE': {'Berlin': {'view': 1}}}


# Problem 29 — Challenge: combine partial aggregates

Suppose data is processed in independent chunks. Each worker returns a regular dictionary of counts.

Write a reducer using `defaultdict(int)` to merge any number of partial count dictionaries.

In [31]:
partials = [
    {"a": 3, "b": 1},
    {"b": 4, "c": 2},
    {"a": 2, "d": 7},
]


def merge_count_dicts(partials):
    merged = defaultdict(int)

    for partial_counts in partials:
        for key, count in partial_counts.items():
            merged[key] += count

    return dict(merged)


merged = merge_count_dicts(partials)
print(merged)

assert merged == {
    "a": 5,
    "b": 5,
    "c": 2,
    "d": 7,
}

{'a': 5, 'b': 5, 'c': 2, 'd': 7}


# Problem 30 — Capstone: analytics pipeline

You receive page-view events:

```text
(user_id, country, page, seconds)
```

Build all of the following in one pass:

1. views per user;
2. total seconds per user;
3. unique pages per user;
4. users per country;
5. total views per `(country, page)`.

Then return a plain dictionary containing all metrics.

### Constraints
- Use appropriate `defaultdict` factories.
- Do not perform a second pass over the original events.
- Avoid duplicate users in `users_per_country`.

In [32]:
page_views = [
    ("u1", "BG", "/home", 5.0),
    ("u1", "BG", "/docs", 12.0),
    ("u2", "BG", "/home", 3.0),
    ("u3", "DE", "/home", 7.0),
    ("u1", "BG", "/home", 2.0),
    ("u3", "DE", "/pricing", 8.0),
]


def analytics_pipeline(events):
    views_per_user = defaultdict(int)
    seconds_per_user = defaultdict(float)
    pages_per_user = defaultdict(set)
    users_per_country = defaultdict(set)
    views_per_country_page = defaultdict(lambda: defaultdict(int))

    for user_id, country, page, seconds in events:
        views_per_user[user_id] += 1
        seconds_per_user[user_id] += seconds
        pages_per_user[user_id].add(page)
        users_per_country[country].add(user_id)
        views_per_country_page[country][page] += 1

    return {
        "views_per_user": dict(views_per_user),
        "seconds_per_user": dict(seconds_per_user),
        "pages_per_user": {
            user_id: set(pages)
            for user_id, pages in pages_per_user.items()
        },
        "users_per_country": {
            country: set(users)
            for country, users in users_per_country.items()
        },
        "views_per_country_page": {
            country: dict(page_counts)
            for country, page_counts in views_per_country_page.items()
        },
    }


analytics = analytics_pipeline(page_views)
pprint(analytics)

assert analytics["views_per_user"] == {
    "u1": 3,
    "u2": 1,
    "u3": 2,
}
assert analytics["seconds_per_user"]["u1"] == 19.0
assert analytics["pages_per_user"]["u1"] == {"/home", "/docs"}
assert analytics["users_per_country"]["BG"] == {"u1", "u2"}
assert analytics["views_per_country_page"]["BG"]["/home"] == 3

{'pages_per_user': {'u1': {'/home', '/docs'},
                    'u2': {'/home'},
                    'u3': {'/home', '/pricing'}},
 'seconds_per_user': {'u1': 19.0, 'u2': 3.0, 'u3': 15.0},
 'users_per_country': {'BG': {'u2', 'u1'}, 'DE': {'u3'}},
 'views_per_country_page': {'BG': {'/docs': 1, '/home': 3},
                            'DE': {'/home': 1, '/pricing': 1}},
 'views_per_user': {'u1': 3, 'u2': 1, 'u3': 2}}


# Additional mini-examples

The following short examples reinforce common `defaultdict` patterns.

In [33]:
# Mini-example A: boolean default
flags = defaultdict(bool)
assert flags["feature_x"] is False

# Mini-example B: empty string default
labels = defaultdict(str)
labels["prefix"] += "API"
assert labels["prefix"] == "API"

# Mini-example C: fresh dict per key
metadata = defaultdict(dict)
metadata["job1"]["status"] = "running"
metadata["job2"]["status"] = "queued"
assert metadata["job1"] is not metadata["job2"]

# Mini-example D: set membership groups
tags = defaultdict(set)
tags["python"].update({"language", "backend"})
tags["python"].add("language")
assert tags["python"] == {"language", "backend"}

# Mini-example E: key-presence side effect
probe = defaultdict(int)
_ = probe["created"]
assert "created" in probe

# Mini-example F: no side effect with get()
probe2 = defaultdict(int)
_ = probe2.get("not_created")
assert "not_created" not in probe2

print("All mini-examples passed.")

All mini-examples passed.


# Testing checklist for `defaultdict` code

When reviewing code that uses `defaultdict`, ask:

- Is the factory callable with no arguments?
- Does each missing key need a **fresh** mutable value?
- Could a read accidentally insert a new key?
- Would `.get()` be safer for read-only access?
- Is a normal `dict` clearer after construction?
- Should nested `defaultdict` values also be converted before serialization?
- Are missing keys truly normal, or should they raise `KeyError`?
- Are zero/empty values meaningful data or merely construction defaults?
- Can the factory itself fail or have side effects?
- Is the chosen factory (`int`, `list`, `set`, `dict`, custom function) aligned with the invariant?

# Final synthesis exercise

Without looking at the earlier solutions, choose the most appropriate factory for each situation:

1. Counting requests per endpoint.
2. Collecting unique IPs per endpoint.
3. Collecting ordered error messages per service.
4. Building nested numeric totals by region and product.
5. Assigning `"unknown"` to missing attributes.
6. Tracking a fresh record with fields `count`, `first_seen`, and `last_seen`.
7. Building an arbitrarily deep configuration tree.

Suggested answers:

1. `int`
2. `set`
3. `list`
4. `lambda: defaultdict(float)`
5. `lambda: "unknown"`
6. a custom no-argument function returning a fresh dictionary
7. a recursive factory

# Summary

`defaultdict` is most powerful when the missing-key behavior is not just a convenience but a stable invariant of the structure.

The most reusable patterns are:

```python
defaultdict(int)      # counting
defaultdict(float)    # numeric accumulation
defaultdict(list)     # ordered grouping
defaultdict(set)      # unique grouping
defaultdict(dict)     # fresh mapping per key
defaultdict(custom_factory)
```

For nested data:

```python
defaultdict(lambda: defaultdict(int))
```

For read-only queries, remember:

```python
d.get(key)
```

does not create a key, while:

```python
d[key]
```

can create one.

That mutation-on-read behavior is one of the most important advanced details to test explicitly.